<a href="https://colab.research.google.com/github/anu-bhav-18/Translation_model_transformer/blob/main/Conditional_Diffusion_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, max_seq_len, d_model):
        super(PositionalEncoding, self).__init__()

        # Create a matrix of [max_seq_len, d_model]
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        # Vectorized implementation of the PE formula
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term) # Even indices
        pe[:, 1::2] = torch.cos(position * div_term) # Odd indices

        # Register as a buffer (so it moves to GPU with the model but isn't a trained parameter)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [seq_len, batch_size, d_model]

        # 1. Slice PE to match the actual seq_len of x
        # 2. Unsqueeze(1) to add the batch dimension for broadcasting

        pe_slice = self.pe[:x.size(0), :].unsqueeze(1)
        #print(f"x shappe:-{x.shape}, \n pe:-{pe_slice.shape}")

        return x + pe_slice


In [9]:
import torch
import torch.nn as nn
from torch.nn import Transformer

class Model(nn.Module):
  def __init__(self,d_model,n_head,n_encoder,n_decoder,d_hidden,seq_len,vocab_size ):
    super(Model,self).__init__()

    self.embeding = nn.Embedding(vocab_size,d_model)
    self.pe = PositionalEncoding(max_seq_len=seq_len,d_model= d_model)
    self.transfomer = Transformer(d_model=d_model,
                                  nhead=n_head,
                                  num_decoder_layers=n_decoder,
                                  num_encoder_layers=n_encoder,
                                  dim_feedforward=d_hidden)

    self.fc_out = nn.Linear(d_model, vocab_size)


  def forward(self,src,tgt,src_mask,tgt_mask):
    src = self.embeding(src)
    src = self.pe(src)

    tgt = self.embeding(tgt)
    tgt = self.pe(tgt)

    output = self.transfomer(src,tgt,src_mask,tgt_mask)
    output = self.fc_out(output)

    return output


In [10]:
import zipfile

with zipfile.ZipFile(file="/content/Dataset_English_Hindi.csv.zip",mode="r") as ref:
  ref.extractall("dataset.csv")

In [11]:
import pandas as pd
from torch.utils.data import Dataset

class TextTranslatinoDataset(Dataset):
  def __init__(self,file_path,tokenizer,max_len,pad_id,eos_id,bos_id ,src_col = "English",tgt_col = "Hindi") -> None:
    super(TextTranslatinoDataset,self).__init__()
    self.file_path = file_path
    self.max_len = max_len
    self.tokenizer = tokenizer
    self.src_col = src_col
    self.pad_id = pad_id
    self.eos_id =eos_id
    self.bos_id = bos_id
    self.tgt_col = tgt_col
    self.data = pd.read_csv(file_path)

  def encode(self,text):
    text_token = self.tokenizer.encode(text).ids
    text_token = [self.bos_id] + text_token + [self.eos_id]
    if len(text_token)< self.max_len:
      text_token = text_token + [self.pad_id]*(self.max_len-len(text_token))
    else:
      text_token = text_token[:self.max_len]
    return text_token

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    src_text = str(self.data.iloc[index][self.src_col])
    tgt_text = str(self.data.iloc[index][self.tgt_col])

    src_token = self.encode(src_text)
    tgt_token = self.encode(tgt_text)

    src_tensor,tgt_tensor = torch.tensor(src_token) , torch.tensor(tgt_token)
    return src_tensor,tgt_tensor


In [12]:
import pandas as pd

df  = pd.read_csv("/content/dataset.csv/Dataset_English_Hindi.csv")
df = df.dropna()

df['English'] = df['English'].astype(str)
df['Hindi'] = df['Hindi'].astype(str)

for line in df['Hindi'].tolist():
  if type(line)==str:
    pass
  else :
    print(type(line))


corpus  = df["English"].tolist() + df["Hindi"].tolist()
#type(corpus)


In [13]:
from tokenizers import models,pre_tokenizers,trainers,Tokenizer

tokenizer = Tokenizer(model=models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(vocab_size=3000,special_tokens=["[UNK]","[PAD]","[SOS]","[EOS]","[BOS]"])

tokenizer.train_from_iterator(corpus,trainer=trainer)
tokenizer.save("eng_hindi_token.json")


In [14]:
pad_id = 1
eos_id = 3
bos_id = 4
sos_id = 2
max_len =20
batch_size =32
d_model = 512
n_head =8
d_hidden =2048
n_encoder =6
n_decoder=6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
from torch.utils.data import DataLoader
dataset = TextTranslatinoDataset(file_path="/content/dataset.csv/Dataset_English_Hindi.csv",
                                 tokenizer= tokenizer,
                                 max_len=max_len,
                                 pad_id=pad_id,
                                 eos_id=eos_id,
                                 bos_id=bos_id )
dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=True)
print(len(dataloader))

4078


In [16]:
from torch.nn import DataParallel
model = Model(d_model,
              n_head,
              n_encoder,
              n_decoder,
              d_hidden,
              max_len,
              tokenizer.get_vocab_size()
              ).to(device)
if torch.cuda.device_count()>1:
  model = DataParallel(model)
  print("Using Data Parallel:-{ torch.cuda.device_count()} devices")
else:
  print(f"not using data parellel:-{ torch.cuda.device_count()} devices")

print(model)



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


not using data parellel:-1 devices
Model(
  (embeding): Embedding(3000, 512)
  (pe): PositionalEncoding()
  (transfomer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleL

In [17]:
for bid,(x,y) in enumerate(dataloader):
  print(bid,x.shape,y.shape)
  break

0 torch.Size([32, 20]) torch.Size([32, 20])


In [18]:
def train(model,loader,epoch,optimizer,loss_fn,device):
  model.train()
  total_loss =[]
  for i in range(epoch):
    total_loss =0
    for bidx,(eng_sen,hindi_sen) in enumerate(loader):
      eng_sen,hindi_sen = eng_sen.to(device),hindi_sen.to(device)
      src = eng_sen.transpose(0,1)
      tgt = hindi_sen.transpose(0,1)

      tgt_input = tgt[:-1,:]
      tgt_target = tgt[1:,:]

      tgt_mask = model.transfomer.generate_square_subsequent_mask(tgt_input.size(0)).to(device)
      optimizer.zero_grad()
      model_output = model(src,tgt_input,src_mask= None,tgt_mask= tgt_mask)
      model_output = model_output.view(-1, model_output.shape[-1])
      tgt_target = tgt_target.reshape(-1)

      loss = loss_fn(model_output,tgt_target)
      loss.backward()
      optimizer.step()
      total_loss+=loss.item()

    print(f"Epoch:-{i+1},Loss:-{total_loss/len(loader)}")



In [20]:
lr = 0.003
epoch = 5
loss_fn = nn.CrossEntropyLoss()
opitimzer = torch.optim.Adam(model.parameters(),lr=lr)
train(model,dataloader,epoch,opitimzer,loss_fn,device)

Epoch:-1,Loss:-5.833107093083268
Epoch:-2,Loss:-5.8302775528455495
Epoch:-3,Loss:-5.829909394452711
Epoch:-4,Loss:-5.829661014560626
Epoch:-5,Loss:-5.829199539206085


In [36]:
def translate_sentence(model, sentence, tokenizer, device, max_len):
    model.eval()

    # 1. Tokenize and prepare source tensor
    # We use your existing encode method from the Dataset class
    src_tokens = [dataset.bos_id] + tokenizer.encode(sentence).ids + [dataset.eos_id]
    src_tensor = torch.tensor(src_tokens).unsqueeze(1).to(device) # Shape: [seq_len, 1]

    # 2. Initialize target with BOS token
    tgt_tokens = [dataset.bos_id]

    for i in range(max_len):
        tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(1).to(device)

        # 3. Generate causal mask for decoder
        tgt_mask = model.transfomer.generate_square_subsequent_mask(len(tgt_tokens)).to(device)

        with torch.no_grad():
            # Forward pass
            output = model(src_tensor, tgt_tensor, src_mask=None ,tgt_mask=tgt_mask)

            # 4. Get the last predicted token (highest probability)
            best_next_token = output.argmax(2)[-1, :].item()


        tgt_tokens.append(best_next_token)

        # 5. Stop if model predicts EOS token
        if best_next_token == dataset.eos_id:
            break
    print(f"Start:-{tgt_tokens}")

    # 6. Convert IDs back to words
    return tokenizer.decode(tgt_tokens)


In [37]:
sentence = "Hello ,how are you?"
token = translate_sentence(model,sentence,tokenizer,device,max_len)
print(token)


Start:-[4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

